# **POD-NN**

POD-NN is a strategy that allows not to rely on affinity on the online stage: the projection stage is not performed and thus the speedup is guaranteed, yet having accurate solutions.
The POD-NN algorithm relies on two stages:
1. a POD,
2. a training of a Feed-forward Neural Network that predicts the entries of the reduced vector $u_{\mathsf{rb}}$.

As usual, we need **a lot of FOM simulations**. Let us import gedim!

In [ ]:
import numpy as np
import argparse
import os.path
import scipy.sparse
import vtk
from pypolydim import polydim, gedim
from pypolydim.export_vtk_utilities import ExportVTKUtilities
from pypolydim.assembler_utilities import assembler_utilities
import matplotlib.pyplot as plt

import sys
sys.path.insert(1, '../')
import other_utilities as other_ut

In [ ]:
geometry_utilities_config = gedim.GeometryUtilitiesConfig()
geometry_utilities_config.tolerance1_d = 1.0e-6
geometry_utilities_config.tolerance2_d = 1.0e-12
geometry_utilities = gedim.GeometryUtilities(geometry_utilities_config)
mesh_utilities = gedim.MeshUtilities()
vtk_utilities = ExportVTKUtilities()

## The parametric version of the heat conductivity equation

Solving the following equation on square $\bar{\Omega} = [-1, +1] \times [-1, +1]$

$$
\begin{cases}
\nabla \cdot (k_{\mu} \nabla u) = 0 & \text{in } \Omega\\
k_{\mu} \nabla u \cdot n_1 = \mu_2 & \text{in } \Gamma_{down}\\
u = \sin(\mu_3\pi x) & \text{in } \Gamma_{up}\\
k_{\mu} \nabla u \cdot n_2 = 0 & \text{otherwise} 
\end{cases}
$$

where $k = \mu_1$ if $x^2 + y^2 \leq R^2$ and $k = 1$ otherwise. 
The parametric space is $\mathcal P = [0.1, 10] \times [0,1] \times [-1, 1]$.

The problem is _standard_. However, we note a nonlinear dependency of the Dirichlet boundary term over $\Gamma_{up}$.

In [ ]:
# Export folder
export_file_path = "./Export/Test_1"
if not os.path.exists(export_file_path):
    os.makedirs(export_file_path)

# Mesh file path
export_mesh_path = export_file_path + "/Mesh"
if not os.path.exists(export_mesh_path):
    os.makedirs(export_mesh_path)

# Solution file path
export_solution_path = export_file_path + "/Solution"
if not os.path.exists(export_solution_path):
    os.makedirs(export_solution_path)

mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D.triangular_simple_importer
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes.fem_pcc
import_mesh_folder = "../Meshes/Mesh3"
method_order = 1

Let us define the High Fidelity Simulation Parameters and import the mesh.

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.import_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          import_mesh_folder,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

In [ ]:
vtk_utilities.export_mesh(export_mesh_path, mesh)
other_ut.plot_mesh(mesh)

Let us create the space

In [ ]:
#### Labels \Gamma_down = 1 (neumann mu_2) , \Gamma_side = 2 (neumann 0) and \Gamma_top = 3 (dirichlet 0)
 
info_internal = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_internal.marker = 0

info_dirichlet_up = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.strong)
info_dirichlet_up.marker = 3

info_neumann_down = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.weak)
info_neumann_down.marker = 1

info_neumann_none = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_neumann_none.marker = 2

boundary_info = {
    0: info_internal,
    1: info_neumann_down,
    2: info_neumann_none,
    3: info_dirichlet_up
}

In [ ]:
mesh_connectivity_data = polydim.pde_tools.mesh.MeshMatricesDAO_mesh_connectivity_data(mesh)

trial_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)
test_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, method_order)

dof_manager = polydim.pde_tools.do_fs.DOFsManager()

trial_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
trial_dofs_data = dof_manager.create_do_fs_2_d(trial_mesh_dofs_info, mesh_connectivity_data)
test_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(trial_reference_element_data, mesh, boundary_info)
test_dofs_data = dof_manager.create_do_fs_2_d(test_mesh_dofs_info, mesh_connectivity_data)

### **Assemble the system**
We can assemble only the parts that are $\mu-$ independent (together with the inner product matrix!). Namely, the Dirichlet term needs to be assembled later on. It is non-affine and nonlinear w.r.t to the parameter $\boldsymbol\mu$!


In [ ]:
R = 0.5

def omega_1(x, y, z):
    if (x * x + y * y) <= (R * R + 1.0e-13):
        return 1.0
    return 0.0

A_omega_1 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_diffusion_operator(geometry_utilities,
                                                                                      mesh,
                                                                                      mesh_geometric_data,
                                                                                      trial_dofs_data,
                                                                                      test_dofs_data,
                                                                                      trial_reference_element_data,
                                                                                      test_reference_element_data,
                                                                                      omega_1)
A_1 = other_ut.make_np_sparse(A_omega_1.operator_dofs)
A_1_D = other_ut.make_np_sparse(A_omega_1.operator_strong)

def omega_2(x, y, z):
    if (x * x + y * y) <= (R * R + 1.0e-13):
        return 0.0
    return 1.0

A_omega_2 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_diffusion_operator(geometry_utilities,
                                                                                      mesh,
                                                                                      mesh_geometric_data,
                                                                                      trial_dofs_data,
                                                                                      test_dofs_data,
                                                                                      trial_reference_element_data,
                                                                                      test_reference_element_data,
                                                                                      omega_2)
A_2 = other_ut.make_np_sparse(A_omega_2.operator_dofs)
A_2_D = other_ut.make_np_sparse(A_omega_2.operator_strong)

#### inner product  
# (||u||^2) + ||grad(u)||^2

def inner_react(x, y, z):
    return 1.0

A_l_2 = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_reaction_operator(geometry_utilities,
                                                                                      mesh,
                                                                                      mesh_geometric_data,
                                                                                      trial_dofs_data,
                                                                                      test_dofs_data,
                                                                                      trial_reference_element_data,
                                                                                      test_reference_element_data,
                                                                                      inner_react)
A_L2 = other_ut.make_np_sparse(A_l_2.operator_dofs)

inner_product = A_1 + A_2 + A_L2  ######## norm
inner_product = A_1 + A_2  ######## semi-norm (equivalent)

def weak_term_function(marker, x, y, z):
    match marker:
        case 1:
            return 1.0
        case _:
            raise ValueError("not valid marker", marker)

weak_term = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_weak_term(geometry_utilities,
                                                                             mesh,
                                                                             mesh_geometric_data,
                                                                             trial_mesh_dofs_info,
                                                                             test_dofs_data,
                                                                             trial_reference_element_data,
                                                                             test_reference_element_data,
                                                                             weak_term_function)
######## DIRICHLET CANNOT BE ASSEMBLED NOW #########################

Let us define the training set for the POD

In [ ]:
### define the training set

snapshot_num = 300
mu1_range = [0.1, 10.]
mu2_range = [-1., 1.]
mu3_range = [-1., 1.]
P = np.array([mu1_range, mu2_range, mu3_range])

training_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(snapshot_num, P.shape[0]))




We can now proceed with the snapshot matrix creation. However, we need to be careful: the problem is not affine in the parameters and we need to assemble the Dirichlet term for each parametric instance.

In [ ]:
############## DIRICHLET VARYING WRT mu_3 #####################
def strong_solution_function(marker, x, y, z):  
    return np.sin(mu_3*np.pi*x) 

#### snapshot matrix creation
thetaA1 = 1
snapshot_matrix = []

tol = 1. - 1e-7
N_max = 10

num_shap = 0
for mu in training_set:
  thetaA2 = mu[0]
  thetaf1 = mu[1]
  mu_3 = mu[2]
  if (num_shap % 30) == 0:
      print("Snapshot", num_shap, " /", snapshot_num, ": ", thetaA2, thetaf1, mu_3)
  
  #### the problem is not affine: I have to assemble in this stage!! ###
  u_D = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function)

  f1_D = A_1_D @ u_D
  f2_D = A_2_D @ u_D

  stiffness = thetaA1*A_1 + thetaA2*A_2
  weakTerm_down = thetaf1*weak_term
  Dirichlet_contribution = thetaA1*f1_D + thetaA2*f2_D
  
  f = weakTerm_down - Dirichlet_contribution
  
  snapshot = scipy.sparse.linalg.spsolve(stiffness, f)
  
  # if you do not want to plot comment
  if (num_shap % 30) == 0:
      proj_full_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          snapshot,
                                                                                          u_D)
      fig = other_ut.plot_solution(mesh, proj_full_on_cell0Ds.numeric_solution)
  
  snapshot_matrix.append(np.copy(snapshot))
  num_shap += 1

snapshot_matrix = np.array(snapshot_matrix) 

  

Let us build and analyze the covariance matrix.

In [ ]:
### covariance matrix

C = snapshot_matrix @ inner_product @ np.transpose(snapshot_matrix) ## metti inner product

# VM, L, VMt = np.linalg.svd((C))

L_e, VM_e = np.linalg.eig(C)
eigenvalues = []
eigenvectors = []


#### check

for i in range(len(L_e)):
  eig_real = L_e[i].real
  eig_complex = L_e[i].imag
  assert np.isclose(eig_complex, 0.)
  eigenvalues.append(eig_real)
  eigenvectors.append(VM_e[i].real)


total_energy = sum(eigenvalues)
retained_energy_vector = np.cumsum(eigenvalues)
relative_retained_energy = retained_energy_vector/total_energy


if all(flag==False for flag in relative_retained_energy>= tol):
  N = N_max
else:
  N = np.argmax(relative_retained_energy >= tol) + 1

print(N)
print(relative_retained_energy)

And now let us build the basis functions and $\mathbb B$.

In [ ]:
# Create the basis function matrix
basis_functions = []
for n in range(N):
  eigenvector =  eigenvectors[n]
  basis = np.transpose(snapshot_matrix)@eigenvector
  norm = np.sqrt(np.transpose(basis) @ inner_product @ basis) ## metti inner product
  basis /= norm
  basis_functions.append(np.copy(basis))

basis_functions = np.transpose(np.array(basis_functions))


If we want to perform standard ROMs we still need to assemble the system.

**Can we asseble it?**

In [ ]:
########## ASSEMBLE WHAT I CAN ##### STILL OFFLINE
reduced_stiff1 = np.transpose(basis_functions) @ A_1 @ basis_functions
reduced_stiff2 = np.transpose(basis_functions) @ A_2 @ basis_functions
reduced_w =  np.transpose(basis_functions) @ weakTerm_down


For each new parameter I have to assemble the Dirichlet term, once again.

In [ ]:
########### I CANNOT DO THAT ################ STILL ONLINE??? WE NEED THE PARAMETER
thetaA2 = 2.
thetaf1 = 0.8
mu_3 = 1.
  

In [ ]:
#### the problem is not affine: I have to assemble in this stage!! ###

Dirichlet_top = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function)
f1_D = A_1_D @ Dirichlet_top
f2_D = A_2_D @ Dirichlet_top
r_f1_D = np.transpose(basis_functions) @ f1_D
r_f2_D = np.transpose(basis_functions) @ f2_D


**Solve linear system for a new $\mu$**


In [ ]:
reduced_rhs = thetaA1*reduced_stiff1 + thetaA2*reduced_stiff2
reduced_lhs = thetaf1*reduced_w - (thetaA1*r_f1_D + thetaA2*r_f2_D)

In [ ]:
#####solve 
reduced_solution = np.linalg.solve(reduced_rhs, reduced_lhs)
print(reduced_solution)

In [ ]:
###### plot #######
proj_reduced_solution = basis_functions @ reduced_solution
proj_full_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          proj_reduced_solution,
                                                                                          Dirichlet_top)
other_ut.plot_solution(mesh, proj_full_on_cell0Ds.numeric_solution)

In [ ]:
stiffness = thetaA1*A_1 + thetaA2*A_2
weakTerm_down = thetaf1*weak_term
f = weakTerm_down - (thetaA1*A_1_D + thetaA2*A_2_D) @ Dirichlet_top
  
full_solution = scipy.sparse.linalg.spsolve(stiffness, f)

In [ ]:
proj_full_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          full_solution,
                                                                                          Dirichlet_top)
other_ut.plot_solution(mesh, proj_full_on_cell0Ds.numeric_solution)

Let us comment a bit on the error analysis and the _speed up_.

In [ ]:
### compute error
import time

abs_err = []
rel_err = []
testing_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(100, P.shape[0]))
speed_up = []

print("Computing error and speedup analysis")

for mu in testing_set:
  
  thetaA2 = mu[0]
  thetaf1 = mu[1]
  mu_3 = mu[2]
  
  #### the problem is not affine: I have to assemble in this stage!! ###
  start_assembling = time.time()
  Dirichlet_top = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function) 
  f1_D = A_1_D @ Dirichlet_top
  f2_D = A_2_D @ Dirichlet_top
  r_f1_D = np.transpose(basis_functions) @ (A_1_D @ Dirichlet_top)
  r_f2_D = np.transpose(basis_functions) @ (A_2_D @ Dirichlet_top)
  time_assembling =  time.time() - start_assembling

  ##### full #####
  stiffness = thetaA1*A_1 + thetaA2*A_2
  weakTerm_down = thetaf1*weak_term
  f = weakTerm_down - (thetaA1*A_1_D + thetaA2*A_2_D) @ Dirichlet_top
  
  
  start_fom = time.time()
  full_solution = scipy.sparse.linalg.spsolve(stiffness, f)
  time_fom = time.time() - start_fom

  #### reduced #####

  reduced_rhs = thetaA1*reduced_stiff1 + thetaA2*reduced_stiff2
  reduced_lhs = thetaf1*reduced_w - (thetaA1*r_f1_D + thetaA2*r_f2_D)
  
  start_rom = time.time()
  reduced_solution = np.linalg.solve(reduced_rhs, reduced_lhs)
  time_rom = time.time() - start_rom
  
  speed_up.append(time_fom/(time_rom + time_assembling))
  
  proj_reduced_solution = basis_functions@reduced_solution

  ### computing error

  error_function = full_solution - proj_reduced_solution
  error_norm_squared_component = np.transpose(error_function) @ inner_product @ error_function
  absolute_error = np.sqrt(abs(error_norm_squared_component))
  abs_err.append(absolute_error)
  
  full_solution_norm_squared_component = np.transpose(full_solution) @  inner_product @ full_solution
  relative_error = absolute_error/np.sqrt(abs(full_solution_norm_squared_component))
  rel_err.append(relative_error)
  

In [ ]:
print("avarege relative error = ", np.mean(rel_err) )
print("avarege absolute error = ", np.mean(abs_err) )
print("avarege speed_up = ", np.mean(speed_up) )

The speed up is quite small for a linear problem. Let understand the role of POD-NN in this setting. 

We want to use a feed-forward NN. Let us define the Class Net with pytorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

mu_dim = P.shape[0]
basis_dim = N 
input_dim = mu_dim
output_dim = basis_dim
nodes = 30

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, nodes) 
        self.fc2 = nn.Linear(nodes, nodes)
        self.fc3 = nn.Linear(nodes, nodes)
        self.fc4 = nn.Linear(nodes, nodes)
        self.fc5 = nn.Linear(nodes, output_dim)
        self.tanh = nn.Tanh()
        # self.apply(self._init_weights)


    def forward(self, x):  ### Forward law ----> prediction
        x = self.tanh(self.fc1(x))   
        x = self.tanh(self.fc2(x))
        x = self.tanh(self.fc3(x))
        x = self.tanh(self.fc4(x))
        x = self.fc5(x)
        return x

In [ ]:
seed_num = 31
torch.manual_seed(seed_num)
net = Net()
torch.set_default_dtype(torch.float32)

my_loss = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
epoch_max = 500000
epoch = 0
tol = 1e-5
loss = 1.

We need to prepare the outputs to train the NN. Indeed, our goal is to define 
$$
\boldsymbol \pi(\boldsymbol \mu) = \underline{u}_{\mathsf{rb}}^{NN}(\boldsymbol \mu).
$$
 Namely, our inputs are the parameters of the training set and the output is the Galerkin projection of the snapshots of the training set.
 The output is of the form $\underline{u}_{\mathsf{rb}}$ where:
 $$
 \mathbb B \underline{u}_{\mathsf{rb}}(\boldsymbol \mu) = \mathbb P^{\boldsymbol \mu}u_{{\delta}}(\boldsymbol \mu), \quad \quad (1)
 $$
 where $\mathbb P^{\boldsymbol \mu} = \mathbb B \mathbb X_{N}^{-1} \mathbb B^T\mathbb X_{N_{\delta}}$ is the reduced vector related to the Galrkin projector, i.e. the best approximation of $u_\delta$ in $V_N$ w.r.t. the inner-product defined by the matrix $X_\delta$.

 Instead of computing the inverse of $\mathbb X_{N} = \mathbb B^T \mathbb X_{{\delta}} \mathbb B$ we solve the following system:
 $$
 \mathbb B^T \mathbb X_{{\delta}} \mathbb B \underline{u}_{\mathsf {rb}}(\boldsymbol \mu) =
 \mathbb X_{N} \mathbb B \underline{u}_{\mathsf {rb}}(\boldsymbol \mu)
  \mathbb B^T \mathbb X_{{\delta}} u_{{\delta}}(\boldsymbol \mu)
 $$
 to find $u_{\mathsf{rb}}(\boldsymbol \mu)$ for each snapshot.

In this way we are taking the vector of the reduced solution related to the parameter $\boldsymbol \mu$ **without solving the reduced system**, thanks to the relation (1). This element is the closest element (the best choice) to $u_{\delta}$ in the norm of the problem.

In [ ]:
####### training set ########
reduced_inner_product = np.transpose(basis_functions) @ inner_product @ basis_functions
x_train = torch.tensor(np.float32(training_set))
y_train = []


for i in range(snapshot_matrix.shape[0]):
  
  snapshot_to_project = snapshot_matrix[i]
  
  projected_snapshot = np.linalg.solve(reduced_inner_product, np.transpose(basis_functions)@inner_product@snapshot_to_project)
  
  y_train.append(projected_snapshot)

y_train = np.float32(y_train)
y_train = torch.tensor(y_train)


Let us train our neural network!

In [ ]:
while loss >= tol and epoch < epoch_max:
  epoch = epoch + 1
  optimizer.zero_grad()
          
  ## compute output
  output = net(x_train)
  
          
  loss = my_loss(output, y_train)
  if epoch >= 20000:
    optimizer.param_groups[0]['lr'] = 0.0001  
  #compute the gradients
  loss.backward()
  # optimizer update
  optimizer.step()
  if (epoch % 2000) == 0:
    print("epoch", epoch, 'loss', loss.item(), 'lr', optimizer.param_groups[0]['lr'] )
              

Let us compute a specific instance of the problem! Namely we compute $\boldsymbol \pi (\boldsymbol \mu_{test})$.

**What is the output?**

Let us compare it with the full solution.

**What do I have to do?**

In [ ]:
x_test = [[6., .1, 1.]]
x_test = np.float32(x_test)
x_test = torch.tensor(x_test)

reduced_solution = np.asarray(net(x_test).detach().numpy())[0]

print(reduced_solution)

In [ ]:
nn_proj_reduced_solution = basis_functions @ reduced_solution
mu = x_test[0]
thetaA2 = mu[0].item()
thetaf1 = mu[1].item()
mu_3 = mu[2].item()
thetaA1 = 1
Dirichlet_top = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function) 
 
proj_full_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          nn_proj_reduced_solution,
                                                                                          Dirichlet_top)
other_ut.plot_solution(mesh, proj_full_on_cell0Ds.numeric_solution)

In [ ]:
##### full #####

stiffness = thetaA1*A_1 + thetaA2*A_2
weakTerm_down = thetaf1*weak_term
f = weakTerm_down - (thetaA1*A_1_D + thetaA2*A_2_D) @ Dirichlet_top
full_solution = scipy.sparse.linalg.spsolve(stiffness, f)

proj_full_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                                          trial_dofs_data,
                                                                                          full_solution,
                                                                                          Dirichlet_top)
other_ut.plot_solution(mesh, proj_full_on_cell0Ds.numeric_solution)

Let us perform an error analysis and comment on the speed up!

In [ ]:
### compute error
import time

abs_err = []
rel_err = []
testing_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(100, P.shape[0]))
speed_up = []

print("Computing error and speedup analysis")

for mu in testing_set:
  
  thetaA2 = mu[0]
  thetaf1 = mu[1]
  mu_3 = mu[2]
  
  #### I DO NOT NEED THE SOLVE #####
  Dirichlet_top = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       trial_mesh_dofs_info,
                                                                       trial_dofs_data,
                                                                       trial_reference_element_data,
                                                                       strong_solution_function)  
  
  ##### full #####
  stiffness = thetaA1*A_1 + thetaA2*A_2
  weakTerm_down = thetaf1*weak_term
  f = weakTerm_down - (thetaA1*A_1_D + thetaA2*A_2_D) @ Dirichlet_top
  
  
  start_fom = time.time()
  full_solution = scipy.sparse.linalg.spsolve(stiffness, f)
  time_fom = time.time() - start_fom
  
  #### reduced #####

  x_test = [[mu[0], mu[1], mu[2]]]
  x_test = np.float32(x_test)
  x_test = torch.tensor(x_test)

  start_rom = time.time()
  reduced_solution = np.asarray(net(x_test).detach().numpy())[0]
  time_rom = time.time() - start_rom
  
  speed_up.append(time_fom/(time_rom))
  
  proj_reduced_solution = basis_functions@reduced_solution
  # gedim.PlotSolution(mesh, dofs, strongs, proj_reduced_solution, Dirichlet_top)

  
  ### computing error

  error_function = full_solution - proj_reduced_solution
  error_norm_squared_component = np.transpose(error_function) @ inner_product @ error_function
  absolute_error = np.sqrt(abs(error_norm_squared_component))
  abs_err.append(absolute_error)
  
  full_solution_norm_squared_component = np.transpose(full_solution) @  inner_product @ full_solution
  relative_error = absolute_error/np.sqrt(abs(full_solution_norm_squared_component))
  rel_err.append(relative_error)


In [ ]:
print("avarege relative error = ", np.mean(rel_err) )
print("avarege absolute error = ", np.mean(abs_err) )
print("avarege speed_up = ", np.mean(speed_up) )